# Auditing Fairness/Bias in Classical ML Income Prediction
**Dataset:** UCI Adult (Census Income) | **Models:** classical ML only (no deep learning)

Pipeline: load data -> EDA -> preprocessing -> baseline models -> group fairness audit -> (next) mitigation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

RANDOM_STATE = 42

## 1. Load data

In [ ]:
adult = fetch_openml("adult", version=2, as_frame=True)
df = adult.frame.copy()
print(df.shape)
df.head()

## 2. Data cleaning

In [ ]:
# Convert categoricals to plain object columns (avoids SimpleImputer issues with pandas 'category' dtype)
cat_like = df.select_dtypes(include=["category", "object"]).columns
for c in cat_like:
    df[c] = df[c].astype(object).where(df[c].notna(), np.nan)
    df[c] = df[c].apply(lambda v: v.strip() if isinstance(v, str) else v)

# Adult marks missing values with '?': turn them into real NaN
n_q = int((df[cat_like] == "?").sum().sum())
df[cat_like] = df[cat_like].replace("?", np.nan)
print("'?' placeholders converted to NaN:", n_q)

# Duplicates
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows dropped:", n_dup, "| shape now:", df.shape)

# Binary target: 1 if income > 50K
df["y"] = df["class"].astype(str).str.contains(">50K").astype(int)
print("\nMissing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

**Cleaning decisions (for the report):**
- `?` placeholders converted to NaN, then imputed inside the model pipeline (median for numeric, most frequent for categorical) so imputation is fit on training data only (no leakage).
- Exact duplicate rows dropped.
- `fnlwgt` (census sampling weight) dropped as a feature: not a personal attribute and not meaningful for prediction.
- Sensitive attributes `sex` and `race` kept in the data for auditing; models are trained both with and without them.

## 3. Exploratory data analysis

In [ ]:
print(df.dtypes)
print("\nTarget balance:")
print(df["y"].value_counts(normalize=True).rename({0: "<=50K", 1: ">50K"}))
print("\nNumeric summary:")
df.describe().T

In [ ]:
# Raw income rate (share earning >50K) by sensitive group: the first sign of bias in the data itself
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ["sex", "race"]):
    tbl = df.groupby(col)["y"].agg(["mean", "count"]).sort_values("mean")
    print(tbl.rename(columns={"mean": "rate_>50K"}).round(3), "\n")
    tbl["mean"].plot.barh(ax=ax)
    ax.set_title(f"Share earning >50K by {col}")
    ax.set_xlabel("rate")
plt.tight_layout(); plt.show()

In [ ]:
# Group sizes and per-group class balance (small groups -> noisy fairness estimates)
print(pd.crosstab(df["race"], df["sex"]))
print()
print(pd.crosstab([df["sex"], df["race"]], df["y"], normalize="index").round(3))

In [ ]:
# Feature distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["age"].plot.hist(bins=30, ax=axes[0], title="Age")
df["hours-per-week"].plot.hist(bins=30, ax=axes[1], title="Hours per week")
df["education-num"].plot.hist(bins=16, ax=axes[2], title="Education (years)")
plt.tight_layout(); plt.show()

In [ ]:
# Outliers: capital gain/loss are heavily skewed with many zeros and a cap value (99999)
print((df[["capital-gain", "capital-loss"]] == 0).mean().round(3))
print(df["capital-gain"].describe())
print("Rows with capital-gain == 99999:", int((df["capital-gain"] == 99999).sum()))

In [ ]:
# Correlation of numeric features with the target
num_df = df.select_dtypes(include="number").drop(columns=["fnlwgt"])
corr = num_df.corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=60, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
fig.colorbar(im); plt.tight_layout(); plt.show()

## 4. Preprocessing

In [ ]:
y = df["y"]
sensitive = df[["sex", "race"]].copy()
X = df.drop(columns=["class", "y", "fnlwgt"])

X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y, sensitive, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

def make_preprocessor(drop_cols=()):
    nums = [c for c in num_cols if c not in drop_cols]
    cats = [c for c in cat_cols if c not in drop_cols]
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), nums),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cats),
    ])

## 5. Baseline models
Two variants per model: with sensitive attributes, and without ("fairness through unawareness").

In [ ]:
models = {
    "LogReg": lambda: LogisticRegression(max_iter=1000),
    "RandForest": lambda: RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
}

results, preds = [], {}
for name, mk in models.items():
    for variant, drop in [("with_sensitive", ()), ("without_sensitive", ("sex", "race"))]:
        pipe = Pipeline([("pre", make_preprocessor(drop)), ("clf", mk())])
        pipe.fit(X_train, y_train)
        yhat = pipe.predict(X_test)
        preds[(name, variant)] = yhat
        results.append({"model": name, "variant": variant,
                        "accuracy": accuracy_score(y_test, yhat), "f1": f1_score(y_test, yhat)})
pd.DataFrame(results).round(4)

## 6. Group fairness audit

In [ ]:
def group_metrics(y_true, y_pred, groups):
    rows = []
    for g in groups.unique():
        m = (groups == g).values
        yt, yp = y_true.values[m], y_pred[m]
        pos, neg = (yt == 1).sum(), (yt == 0).sum()
        rows.append({
            "group": g, "n": m.sum(),
            "selection_rate": yp.mean(),                                   # P(yhat=1)
            "TPR": ((yp == 1) & (yt == 1)).sum() / pos if pos else np.nan,  # equal opportunity
            "FPR": ((yp == 1) & (yt == 0)).sum() / neg if neg else np.nan,
            "accuracy": (yp == yt).mean(),
        })
    return pd.DataFrame(rows).set_index("group")

def gaps(tbl):
    return {
        "demographic_parity_diff": tbl["selection_rate"].max() - tbl["selection_rate"].min(),
        "equal_opportunity_diff": tbl["TPR"].max() - tbl["TPR"].min(),
        "FPR_diff": tbl["FPR"].max() - tbl["FPR"].min(),
    }

for (name, variant), yhat in preds.items():
    for attr in ["sex", "race"]:
        tbl = group_metrics(y_test, yhat, s_test[attr])
        print(f"=== {name} | {variant} | {attr} ===")
        print(tbl.round(3))
        print({k: round(v, 3) for k, v in gaps(tbl).items()}, "\n")

## 7. Next steps (team)
- **Preprocessing:** try other imputation, check the `?` handling, add feature notes
- **Models:** tune hyperparameters, add SVM and gradient boosting
- **Fairness eval:** bootstrap confidence intervals + significance tests on the gaps above
- **Mitigation:** reweighing, per-group thresholds, compare accuracy vs fairness trade-off
- **Report/poster:** plots of selection rate / TPR by group before vs after mitigation